# FULL DQN 450K - manual launch and recovery

**12 runs x450,000 selected actions =5.4 million.** V0, V1, history-only and V2; seeds42/123/456. Frozen `dqn_extended_v1_20261024`: ls20/ar25/bp35, round robin, existing rewards, replay, epsilon and milestones.

Upload this notebook to Colab, choose **T4 GPU**, then **Run all**. Upload **dqn_extended_full_source.zip** in Cell2 and approve Google access in Cell3. No Chrome automation or remote branch is used. The real completed75K pilot evidence is bundled and revalidated; the75K pilots are NOT rerun.

Existing3K/10K/75K evidence is preserved. These are new450K runs. Training does not start until source, pilot evidence, policy encoding and a short forced CUDA interruption/recovery test pass. Reserved heldout games are absent from the assets and never run.


## Cell 1 - Setup

Check the T4, show RAM/disk, install the compatible pinned dependencies. Scientific dependencies match the successful T4 pilots. If the runtime has another PyTorch version, install the pilot's CUDA wheel for the fresh training subprocesses; the notebook kernel itself does no training.


In [ ]:
import os,sys,subprocess,hashlib,json,zipfile,uuid,time,shutil,io,signal
from pathlib import Path
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
assert sys.version_info >= (3,12), 'Use Colab Python3.12 or later'
def run_checked(command):
    process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
        text=True,bufsize=1,start_new_session=True)
    try:
        for line in process.stdout: print(line,end='',flush=True)
        status=process.wait()
    except BaseException:
        try:
            os.killpg(process.pid,signal.SIGTERM)
            process.wait(timeout=20)
        except ProcessLookupError: pass
        raise
    if status: raise subprocess.CalledProcessError(status,command)
    return process
import torch
assert torch.cuda.is_available(), 'CUDA is unavailable. Select T4 GPU; there is no CPU fallback.'
assert 'T4' in torch.cuda.get_device_name(0), 'A T4 is required for the frozen experiment hardware.'
run_checked(['nvidia-smi'])
print('RAM:',Path('/proc/meminfo').read_text().splitlines()[:3])
print('VM disk:',shutil.disk_usage('/content'))
run_checked([sys.executable,'-m','pip','install','arc-agi==0.9.9','arcengine==0.9.3',
    'gymnasium==1.4.0','numpy==2.5.4','PyYAML==6.0.3','pytest==8.4.2','filelock==4.2.4',
    'google-api-python-client==2.201.0','google-auth==2.49.0','google-auth-httplib2==0.4.4'])
if str(torch.__version__)!='2.11.0+cu130':
    run_checked([sys.executable,'-m','pip','install','torch==2.11.0','--index-url','https://download.pytorch.org/whl/cu130'])
run_checked([sys.executable,'-c',"import torch; assert str(torch.__version__)=='2.11.0+cu130'; assert torch.cuda.is_available(); assert 'T4' in torch.cuda.get_device_name(0); print('Training Torch:',torch.__version__,'CUDA:',torch.version.cuda)"])


## Cell 2 - Load project source

Choose only **dqn_extended_full_source.zip**. Every bundled file is checked against its manifest. The bundle includes only the six already exposed training/development games and the genuine pilot evidence; no credentials, historical checkpoint directories or heldout game assets.


In [ ]:
from google.colab import files
uploaded=files.upload()
assert len(uploaded)==1, 'Select exactly dqn_extended_full_source.zip'
name,raw=next(iter(uploaded.items()))
EXPECTED_SOURCE_SHA256='3e85adc7cd9af2c61b05bc739d3080ffdcf1e754c11bbd2ed6a6f5a039339b5a'
assert hashlib.sha256(raw).hexdigest()==EXPECTED_SOURCE_SHA256, 'Wrong or changed source ZIP. Use the ZIP delivered with this notebook.'
ROOT=Path('/content')/('arc_dqn_full_'+uuid.uuid4().hex[:10])
ROOT.mkdir(exist_ok=False)
with zipfile.ZipFile(io.BytesIO(raw)) as z:
    for member in z.namelist(): assert (ROOT/member).resolve().is_relative_to(ROOT.resolve())
    z.extractall(ROOT)
manifest=json.loads((ROOT/'FULL_SOURCE_MANIFEST.json').read_text())
for path,expected in manifest['sha256'].items():
    assert hashlib.sha256((ROOT/path).read_bytes()).hexdigest()==expected,path
os.chdir(ROOT)
sys.path.insert(0,str(ROOT))
print('Verified source:',EXPECTED_SOURCE_SHA256)
print('Frozen protocol:',manifest['protocol']['protocol_id'])


## Cell 3 - Google Drive

Approve standard Colab Drive mounting and Google authentication for the storage-quota check. Google may show two approval prompts here. **No further authentication is needed by the queue after setup.** At least4GiB of account storage must be available at launch; reserve512MiB is checked before each attempt. The actual account quota is checked, not just the VM disk. Training stops safely on quota/write errors.

Mount recovery: an existing readable mount is preserved. Otherwise the cell attempts the standard mount twice, using force_remount only on the second attempt. Authorization cancellation stops the cell. This cannot repair an account-side Google Drive service failure; training remains blocked until a readable Drive is available. The source ZIP is unchanged.


In [ ]:
from google.colab import drive,auth
def ensure_drive_mounted(drive_api, mountpoint='/content/drive', ready=None):
    """Reuse a readable mount, or make at most two standard Colab mount calls."""
    if ready is None:
        def ready():
            folder=Path(mountpoint)/'MyDrive'
            try:
                if not os.path.ismount(mountpoint) or not folder.is_dir():
                    return False
                next(folder.iterdir(), None)
                return True
            except OSError:
                return False
    if ready():
        print('Google Drive is already readable; keeping the current mount.')
        return
    last_error=None
    for attempt in range(2):
        print(f'Google Drive mount attempt {attempt+1}/2. Complete the Google authorization prompt.',flush=True)
        try:
            drive_api.mount(mountpoint,force_remount=(attempt==1),timeout_ms=180000)
            if ready():
                print('Google Drive mounted and MyDrive is readable.',flush=True)
                return
            last_error=OSError('Mount returned without a readable MyDrive folder')
        except Exception as exc:
            last_error=exc
            if any(word in str(exc).lower() for word in ('denied','cancelled','canceled','permission')):
                break
        if attempt==0:
            print('Mount did not become ready. Retrying once with Colab force_remount.',flush=True)
    raise RuntimeError(
        'Google Drive is not mounted; training remains stopped. '
        'Use the Colab Files sidebar Mount Drive button and finish Google authorization with the account containing your experiment folder, '
        'then rerun Cell3. If that also fails, reconnect a fresh Colab runtime and rerun setup/source/Drive with the same ZIP. '
        'Do not run the training cell until Drive mounts successfully. '
        f'Last mount error: {last_error}'
    ) from last_error

ensure_drive_mounted(drive)
auth.authenticate_user()
from Experiments.drive_storage import require_space
storage=require_space(4*2**30)
print('Drive account quota:',storage)
FULLROOT=Path('/content/drive/MyDrive/ARC_DQN_extended/FULL_450K_dqn_extended_v1_20261024')
FULLROOT.mkdir(parents=True,exist_ok=True)
probe=FULLROOT/('.write_probe_'+uuid.uuid4().hex)
probe.write_text('verified writable')
assert probe.read_text()=='verified writable'
probe.unlink()
(FULLROOT/'storage_at_setup.json').write_text(json.dumps(storage,indent=2))
LOCAL=Path('/content/ARC_DQN_FULL_WORK')
LOCAL.mkdir(parents=True,exist_ok=True)
shutil.copy2(ROOT/'FULL_SOURCE_MANIFEST.json',FULLROOT/'FULL_SOURCE_MANIFEST.json')
with (FULLROOT/'pip_freeze_latest.txt').open('w') as f:
    subprocess.run([sys.executable,'-m','pip','freeze'],stdout=f,check=True)
print('Permanent results:',FULLROOT)


## Cell 4 - Recovery and preflight

Revalidate the real75K evidence, verify all4102 action indices and reproduce greedy pilot actions from the saved weights on exposed games. Then perform a short actual CUDA process-kill/recovery test with a verified Drive checkpoint. The learner outside `publish()` is checked against the pilot source by AST; only durability handling changed.

On a later session, this repeats only the short preflight, never the75K pilots. Existing full pilot continuation data, if present in its original Drive batch, is checked too. Full-matrix runs start fresh and use their own complete state; they do not depend on loading a pilot replay.


In [ ]:
run_checked([sys.executable,'-m','Experiments.prepare_extended_assets','--include-development'])
tests_receipt=FULLROOT/'TESTS_PASSED.json'
if not tests_receipt.exists() or json.loads(tests_receipt.read_text()).get('source_archive')!=EXPECTED_SOURCE_SHA256:
    run_checked([sys.executable,'-m','pytest','-q','tests/test_full_queue.py','tests/test_dqn_devices.py',
        'tests/test_dqn_extended.py','tests/test_dqn_shared_training.py'])
    tests_receipt.write_text(json.dumps({'valid':True,'source_archive':EXPECTED_SOURCE_SHA256,'torch':'2.11.0+cu130','time':time.time()}))
else:
    print('Reusing passed regression checks for this exact source archive; CUDA recovery still checked for this VM.')
# Independently inspect original complete75K generations when accessible.
pilot_root=Path('/content/drive/MyDrive/ARC_DQN_extended/pilots_20261024T211337Z_f07f3651')
markers=list(pilot_root.glob('v*_seed42_*/resume/step075000/COMPLETE.json')) if pilot_root.exists() else []
for marker in markers:
    run_checked([sys.executable,'-c',"from pathlib import Path; from DQN.extended_train import check_resume; s=check_resume(Path(__import__('sys').argv[1])); assert s['global_step']==75000; print('Verified complete pilot:',s['variant'],s['global_step'])",str(marker.parent)])
print('Original full pilot generations found:',len(markers),'; archival pilot evidence is independently bundled.')
boot=Path('/proc/sys/kernel/random/boot_id').read_text().strip()
proof_file=FULLROOT/'PREFLIGHT_PASSED.json'
reusable=False
if proof_file.exists():
    proof=json.loads(proof_file.read_text())
    expected={p:sha for p,sha in manifest['sha256'].items() if p.startswith(('DQN/','arc_env/'))}
    reusable=proof.get('valid') and proof.get('boot_id')==boot and proof.get('source_sha256')==expected and proof.get('source_archive')==EXPECTED_SOURCE_SHA256
if not reusable:
    preflight=LOCAL/('preflight_'+uuid.uuid4().hex[:8])
    run_checked([sys.executable,'-m','Experiments.full_preflight','--out',str(preflight),
        '--persistent',str(FULLROOT/'preflight')])
    proof=json.loads((preflight/'PREFLIGHT_PASSED.json').read_text())
    proof['source_archive']=EXPECTED_SOURCE_SHA256
    raw=json.dumps(proof,indent=2).encode()
    partial=FULLROOT/'PREFLIGHT_PASSED.json.partial'
    partial.write_bytes(raw)
    assert hashlib.sha256(partial.read_bytes()).digest()==hashlib.sha256(raw).digest()
    partial.replace(proof_file)
QUEUE_COMMAND=[sys.executable,'-u','-m','Experiments.full_queue','--root',str(FULLROOT),
    '--local',str(LOCAL/'training'),'--config',str(ROOT/'Configs/dqn_extended_v1.json')]
print('PREFLIGHT PASSED. Cell5 launches ALL TWELVE450K experiments.')


## Cell 5 - LAUNCH ALL 12 EXPERIMENTS

**This launches the full matrix, not the two-pilot workflow.** The queue uses one T4 sequentially. New runs print `TRAIN v0_seed42: 0/450000`. Progress lines show observed actions and durable actions. `DRIVE COMPLETE ...: 25000` confirms the first complete recovery checkpoint was verified on Drive.

The queue verifies existing checkpoints independently of its progress file, skips completed jobs, and retries recoverable process failures at most twice from the latest verified state. Protocol/version/corruption/quota failures stop safely. No CPU fallback or keep-alive script is used. The estimated full workload is roughly10 GPU hours; Colab Free availability is not guaranteed.


In [ ]:
run_checked(QUEUE_COMMAND)
QUEUE_FINISHED_THIS_SESSION=True
print('ALL12 RUNS VERIFIED COMPLETE. Check Drive progress.json and per-run RUN_COMPLETE.json.')


## RESUME TRAINING - after complete runtime termination

Reconnect to an available T4. Rerun Cells1-4 with the SAME two files, approve Drive if requested, then run this cell instead of Cell5. Paths need no editing. The queue finds the latest valid generation, ignores an unfinished tail, verifies and reconstructs game states, and continues the remaining jobs. Use the recorded package/runtime versions; it fails rather than silently changing a running experiment.

A terminated VM cannot restart itself. There are no quota/session workarounds. Only progress through the latest complete25K checkpoint is guaranteed; work since that checkpoint may be repeated.


In [ ]:
if globals().get('QUEUE_FINISHED_THIS_SESSION'):
    print('The queue already completed in this session; nothing to resume.')
else:
    run_checked(QUEUE_COMMAND)
    QUEUE_FINISHED_THIS_SESSION=True


## Outputs and development evaluation

Permanent folder: **MyDrive/ARC_DQN_extended/FULL_450K_dqn_extended_v1_20261024**.

- `progress.json`: all12 jobs, observed/durable action counts, latest verified checkpoint, throughput and failure status.
- `<variant>/seed<seed>/attempts/<attempt>/resume/stepNNNNNN`: state.pt, replay.npz, history.csv.gz and COMPLETE.json. Latest two valid generations retained across attempts.
- `<variant>/seed<seed>/frozen`: all six75K evaluation milestones.
- `<variant>/seed<seed>/RUN_COMPLETE.json`: independently checked completion receipt.
- Per-attempt logs, final canonical committed training history and software/source manifests.

Development evaluation is separate so training remains the priority. After training, from the loaded project directory run:

`python -m Experiments.prepare_full_development --queue-root /content/drive/MyDrive/ARC_DQN_extended/FULL_450K_dqn_extended_v1_20261024 --staging /content/dqn_full_evaluation_models --out /content/drive/MyDrive/ARC_DQN_extended/FULL_450K_development_evaluation`

This prepares all six milestones and uses the shared evaluator on ft09/vc33/cd82 only. Reserved sb26/sk48/su15 stay untouched. Historical PPO training differences remain limitations; no algorithm-superiority claim follows from unequal historical contracts.
